# 03 — Qualité des données, validation du pipeline, dictionnaire de données

**Projet NutriHFr** · Livrables 2 (Data dictionary) et 3 (Data Quality Report) de l'annexe B

1. Contrôle qualité selon les **8 dimensions** (complétude, unicité, validité, cohérence, intégrité référentielle, exactitude, homogénéité, traçabilité).
2. **Validation du pipeline** : nos apports recalculés sont comparés à ceux fournis par l'Anses.
3. Contrôle de la composition avec **Ciqual** (cohérence énergie / coefficients d'Atwater).
4. Génération du **dictionnaire de données**.
5. Lancement des **tests automatisés** (annexe D).

In [ ]:
# --- Configuration commune à tous les notebooks ------------------------------
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C
from nutrihfr.viz import set_style, savefig, PALETTE, SEX_COLORS
set_style()
if C.MODE == "synthetic" and not C.raw_dir("inca3").exists():   # 1re utilisation : données de démo
    import runpy; runpy.run_path(str(ROOT / "scripts" / "make_synthetic_data.py"), run_name="__main__")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)
print("Mode des données :", C.MODE, "->", C.raw_dir("inca3"))

## 1. Exécution du pipeline complet

In [ ]:
from nutrihfr.pipeline import NutriPipeline
pipe = NutriPipeline()
res = pipe.run()          # recharge, contrôle, nettoie, calcule, écrit la base et docs/data_quality_report.md

## 2. Rapport qualité (8 dimensions)

In [ ]:
q = res["quality"]
display(q.groupby(["dimension", "statut"]).size().unstack(fill_value=0))
q[q["statut"] != "OK"]

In [ ]:
res["exclusions_summary"]

**Lecture** : chaque point d'attention a une règle de traitement documentée (notebook 02, section 4). Le rapport complet est écrit automatiquement dans `docs/data_quality_report.md` à chaque exécution : il est donc toujours à jour avec les données.

## 3. Validation : apports recalculés vs apports fournis par l'Anses

C'est **le test de validation principal** suggéré par la consigne : si notre chaîne (jointure consommation × composition, formule, agrégation par jour puis par individu) est correcte, on doit retrouver les apports publiés.

In [ ]:
res["validation"]

In [ ]:
from nutrihfr.loader import Inca3Loader
prov = res["tables"]["apports_fournis"]
m = res["indiv"][["id_indiv", "energie_kcal", "calcium_mg"]].merge(prov, on="id_indiv", suffixes=("_calc", "_anses"))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, n in zip(axes, ["energie_kcal", "calcium_mg"]):
    ax.scatter(m[f"{n}_anses"], m[f"{n}_calc"], s=8, alpha=0.4, color=PALETTE[0])
    lim = [0, m[[f"{n}_anses", f"{n}_calc"]].max().max()]
    ax.plot(lim, lim, color="#52514e", lw=1, ls="--", label="y = x")
    ax.set(title=n, xlabel="apport fourni (Anses)", ylabel="apport recalculé (pipeline)")
    ax.legend()
plt.tight_layout(); savefig(fig, "03_validation_anses"); plt.show()

**Interprétation** : les points s'alignent sur la diagonale. Les rares écarts viennent des lignes écartées au nettoyage (doublons, quantités implausibles) et des arrondis. *Avec les vraies données, des écarts plus importants peuvent indiquer : une mauvaise colonne de nutriment dans `config.py`, des valeurs « pour la portion » prises pour des valeurs « pour 100 g », ou des apports Anses qui incluent les compléments alimentaires.* Ces écarts se commentent dans le rapport final.

## 4. Composition : contrôle avec Ciqual (coefficients d'Atwater)

Énergie recalculée = 4 × protéines + 4 × glucides + 9 × lipides + 2 × fibres + 7 × alcool (kcal/100 g, règlement UE 1169/2011). On la compare à l'énergie indiquée dans Ciqual.

In [ ]:
from nutrihfr.loader import CiqualLoader
from nutrihfr.intake import atwater_energy
cl = CiqualLoader()
ciq = cl.load()
print(cl.log, "| version :", C.CIQUAL_VERSION)
ciq["energie_atwater"] = atwater_energy(ciq)
ciq["ecart_pct"] = 100 * (ciq["energie_atwater"] - ciq["energie_kcal"]) / ciq["energie_kcal"]
ok = ciq["energie_kcal"] > 20
print(f"{(ciq.loc[ok, 'ecart_pct'].abs() < 10).mean():.1%} des aliments (> 20 kcal) ont un écart < 10 %")
ciq.loc[ok].sort_values("ecart_pct", key=abs, ascending=False)[["libelle", "energie_kcal", "energie_atwater", "ecart_pct"]].head(8)

## 5. Dictionnaire de données

Chaque variable produite par le pipeline est documentée (nom, définition, type, unité, source). Le fichier est écrit dans `docs/data_dictionary.md`.

In [ ]:
DEFS = {
    "id_indiv": ("Identifiant pseudonymisé de l'individu", "", "INCA3 NOIND"),
    "enquete": ("Enquête d'origine", "", "pipeline"),
    "sexe": ("Sexe", "Homme/Femme", "INCA3 sex_PS recodé"),
    "age": ("Âge", "années", "INCA3"),
    "classe_age": ("Classe d'âge du projet", "", "pipeline (18-24, 25-44, 45-64, 65-79)"),
    "classe_age_src": ("Classe d'âge fournie par l'Anses", "code", "INCA3 tage_PS"),
    "poids_kg": ("Poids corporel", "kg", "INCA3"),
    "taille_cm": ("Taille", "cm", "INCA3"),
    "poids_enquete": ("Pondération d'enquête (adultes)", "", "INCA3 pond_indiv_adu_pop2"),
    "n_jours": ("Nombre de journées de consommation retenues", "jours", "pipeline"),
    "id_jour": ("Identifiant de la journée (id_indiv_no_jour)", "", "pipeline"),
    "no_jour": ("Numéro de la journée de recueil", "", "INCA3 nojour"),
    "id_conso": ("Identifiant de la ligne de consommation", "", "pipeline"),
    "code_aliment": ("Code de l'aliment (nomenclature INCA 3)", "", "INCA3"),
    "libelle": ("Libellé de l'aliment", "", "INCA3 nomenclature"),
    "groupe": ("Groupe d'aliments", "", "INCA3 nomenclature"),
    "quantite_g": ("Quantité consommée", "g", "INCA3"),
    "valeur_100g": ("Teneur du nutriment pour 100 g d'aliment", "unité du nutriment", "INCA3 conso-compo-alim"),
    "code_nutriment": ("Code canonique du nutriment", "", "config.NUTRIENTS"),
    "source": ("Origine de la valeur de composition", "", "pipeline"),
    "bmr_kcal": ("Métabolisme de base estimé (Schofield)", "kcal/j", "pipeline"),
    "ei_bmr": ("Rapport apport énergétique / métabolisme de base", "", "pipeline"),
    "seuil_goldberg": ("Seuil de Goldberg (Black 2000) selon le nombre de jours", "", "pipeline"),
    "sous_declarant": ("Sous-déclarant de l'énergie (EI/BMR < seuil)", "booléen", "pipeline"),
}
for n, (lib, unit) in C.NUTRIENTS.items():
    DEFS[n] = (f"{lib} : apport journalier moyen (ou teneur selon la table)", f"{unit}/j", "pipeline")
    DEFS[f"{n}_per1000kcal"] = (f"{lib} pour 1 000 kcal", f"{unit}/1000 kcal", "pipeline")
for k in ("proteines", "glucides", "lipides"):
    DEFS[f"pct_ae_{k}"] = (f"Part de l'apport énergétique due aux {k}", "%", "pipeline")

rows = []
for tname, df in [("individus + apports", res["indiv"]), ("consommations", res["tables"]["consommations"]),
                  ("aliments", res["tables"]["aliments"]), ("composition", res["tables"]["composition"])]:
    for col in df.columns:
        d = DEFS.get(col, ("(à documenter)", "", ""))
        rows.append({"table": tname, "variable": col, "definition": d[0], "type": str(df[col].dtype),
                     "unite": d[1], "source": d[2], "% manquants": round(100 * df[col].isna().mean(), 2)})
dico = pd.DataFrame(rows)
md_txt = "# Dictionnaire de données\n\n_Généré par le notebook 03._\n\n| Table | Variable | Définition | Type | Unité | Source | % manquants |\n|---|---|---|---|---|---|---|\n"
md_txt += "".join(f"| {r.table} | `{r.variable}` | {r.definition} | {r.type} | {r.unite} | {r.source} | {r._7} |\n" for r in dico.itertuples())
(ROOT / "docs" / "data_dictionary.md").write_text(md_txt, encoding="utf-8")
print("Variables non documentées :", dico.loc[dico.definition.str.startswith("(à"), "variable"].tolist())
dico.head(15)

## 6. Tests automatisés (annexe D)

Les tests sont dans `tests/test_pipeline.py` ; ils se lancent depuis la racine du dépôt avec `pytest -v`.

| Test | Contrôle |
|---|---|
| 1 | aucune quantité négative |
| 2 | 100 % des nutriments ont une unité connue |
| 3 | tout aliment consommé a une composition, ou figure dans la table d'exceptions |
| 4 | 200 g de lait à 120 mg/100 g donnent bien 240 mg de calcium |
| 5 | 0 g consommé donne 0 apport |
| 6 | la somme des pondérations est cohérente avec la population de référence |
| 7 | énergie Atwater ≈ énergie Ciqual (< 10 % pour 95 % des aliments) |

In [ ]:
import subprocess
out = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests")],
                     capture_output=True, text=True, cwd=ROOT)
print(out.stdout[-2000:] or out.stderr[-2000:])

➡️ Suite : `04_analyses_Q1_Q6.ipynb`